In [1]:
# =============================================================================
# FINANCIAL HEALTH INDEX (FHI) PREDICTION — WINNING SOLUTION
# =============================================================================
# Competition : Zindi — SME Financial Health Index Prediction
# Target      : Low / Medium / High (multiclass classification)
# Metric      : Macro F1 Score
# Public LB   : 0.8847
# Private LB  : 0.8860
#
# Pipeline:
#   1. Data cleaning & preprocessing (apostrophe fix + full encoding cleanup)
#   2. Feature engineering (log transforms, ratios, missing flags)
#   3. SMOTE minority oversampling per fold
#   4. XGBoost + LightGBM + CatBoost ensemble (Optuna tuned)
#   5. Weighted ensemble + threshold optimization
# =============================================================================

In [2]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from scipy.optimize import minimize
 
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
 
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================
SEED        = 42
N_FOLDS     = 5
N_TRIALS    = 50
TARGET_COL  = "Target"
ID_COL      = "ID"
LABEL_ORDER = ["Low", "Medium", "High"]
 
DATA_DIR    = Path(".")
OUTPUT_DIR  = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
 
# Label encoder — alphabetical sort: High=0, Low=1, Medium=2
le_target = LabelEncoder()
le_target.fit(LABEL_ORDER)
 
print("=" * 60)
print("FINANCIAL HEALTH INDEX PREDICTION — WINNING SOLUTION")
print("=" * 60)
print(f"Target classes : {le_target.classes_}")
print(f"N_FOLDS        : {N_FOLDS}")
print(f"N_TRIALS       : {N_TRIALS}")

FINANCIAL HEALTH INDEX PREDICTION — WINNING SOLUTION
Target classes : ['High' 'Low' 'Medium']
N_FOLDS        : 5
N_TRIALS       : 50


In [4]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("\n1. Loading data...")
train = pd.read_csv(DATA_DIR / "/kaggle/input/datasets/frankmandele/financial-health-prediction/Train.csv")
test  = pd.read_csv(DATA_DIR / "/kaggle/input/datasets/frankmandele/financial-health-prediction/Test.csv")
 
print(f"   Train : {train.shape}")
print(f"   Test  : {test.shape}")
print(f"   Target distribution:\n{train[TARGET_COL].value_counts().to_string()}")


1. Loading data...
   Train : (9618, 39)
   Test  : (2405, 38)
   Target distribution:
Target
Low       6280
Medium    2868
High       470


In [5]:
def clean_and_preprocess(df):
    df = df.copy()

    # -------------------------------------------------------------------------
    # FIX 1: Normalize apostrophe variants (curly → straight)
    # -------------------------------------------------------------------------
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.replace('\u2019', "'", regex=False)  # curly → straight

    # -------------------------------------------------------------------------
    # FIX 2: Normalize case and whitespace across all string columns
    # -------------------------------------------------------------------------
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    # -------------------------------------------------------------------------
    # FIX 3: Standardize all "don't know" variants to a single value
    # -------------------------------------------------------------------------
    dont_know_variants = [
        "Don't know", "Don't Know", "Don't know or N/A",
        "Don't know (Do not show)", "Don?t know / doesn?t apply",
        "Do not know / N\u200e/A", " Do not know / N\u200e/A",
        "Refused"
    ]
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].replace(dont_know_variants, "Unknown")

    # -------------------------------------------------------------------------
    # FIX 4: current_problem_cash_flow — "0" → "No"
    # -------------------------------------------------------------------------
    if "current_problem_cash_flow" in df.columns:
        df["current_problem_cash_flow"] = df["current_problem_cash_flow"].replace({"0": "No"})

    # -------------------------------------------------------------------------
    # FIX 5: owner_age — 99/103 likely survey placeholders → NaN
    # -------------------------------------------------------------------------
    if "owner_age" in df.columns:
        df["owner_age"] = df["owner_age"].apply(
            lambda x: np.nan if pd.notna(x) and x >= 99 else x
        )

    # -------------------------------------------------------------------------
    # FIX 6: keeps_financial_records — unexpected values → Yes
    # -------------------------------------------------------------------------
    if "keeps_financial_records" in df.columns:
        df["keeps_financial_records"] = df["keeps_financial_records"].replace({
            "Yes, always": "Yes",
            "Yes, sometimes": "Yes"
        })

    # -------------------------------------------------------------------------
    # ENCODE: Status columns (Have now / Never had / Used to have)
    # -------------------------------------------------------------------------
    status_map = {
        "Have now": 2,
        "Used to have but don't have now": 1,
        "Never had": 0,
        "Unknown": -1,
    }
    status_cols = [
        "motor_vehicle_insurance", "has_mobile_money", "has_credit_card",
        "has_loan_account", "has_internet_banking", "has_debit_card",
        "medical_insurance", "funeral_insurance",
        "uses_friends_family_savings", "uses_informal_lender"
    ]
    for col in status_cols:
        if col in df.columns:
            df[col] = df[col].map(status_map)  # unmapped → NaN automatically

    # -------------------------------------------------------------------------
    # ENCODE: Binary Yes/No columns
    # -------------------------------------------------------------------------
    binary_map = {"Yes": 1, "No": 0, "Unknown": -1}
    binary_cols = [
        "attitude_stable_business_environment", "attitude_worried_shutdown",
        "compliance_income_tax", "perception_insurance_doesnt_cover_losses",
        "perception_cannot_afford_insurance", "has_cellphone",
        "attitude_satisfied_with_achievement", "keeps_financial_records",
        "perception_insurance_companies_dont_insure_businesses_like_yours",
        "perception_insurance_important", "has_insurance",
        "covid_essential_service", "attitude_more_successful_next_year",
        "problem_sourcing_money", "marketing_word_of_mouth",
        "future_risk_theft_stock", "motivation_make_more_money",
        "current_problem_cash_flow"
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map(binary_map)

    # -------------------------------------------------------------------------
    # ENCODE: owner_sex
    # -------------------------------------------------------------------------
    if "owner_sex" in df.columns:
        df["owner_sex"] = df["owner_sex"].map({"Male": 1, "Female": 0})

    # -------------------------------------------------------------------------
    # ENCODE: offers_credit_to_customers
    # -------------------------------------------------------------------------
    if "offers_credit_to_customers" in df.columns:
        df["offers_credit_to_customers"] = df["offers_credit_to_customers"].map({
            "Yes, always": 2, "Yes, sometimes": 1, "No": 0
        })

    # -------------------------------------------------------------------------
    # ENCODE: country — one-hot (safer than ordinal)
    # -------------------------------------------------------------------------
    if "country" in df.columns:
        df = pd.get_dummies(df, columns=["country"], prefix="country", drop_first=False)

    return df

# Apply to train and test
train_fe = clean_and_preprocess(train.drop(columns=[TARGET_COL, ID_COL]))
test_fe  = clean_and_preprocess(test.drop(columns=[ID_COL]))

# Align columns
train_fe, test_fe = train_fe.align(test_fe, join="left", axis=1, fill_value=0)

# Encode target
y = le_target.transform(train[TARGET_COL])

# Impute
imputer = SimpleImputer(strategy="median")
X      = imputer.fit_transform(train_fe.values)
X_test = imputer.transform(test_fe.values)

feature_names = list(train_fe.columns)

print(f"Features: {len(feature_names)}")
print(f"Feature list: {feature_names}")

# Quick sanity check — no NaN should remain
print(f"\nNaN in X: {np.isnan(X).sum()}")
print(f"NaN in X_test: {np.isnan(X_test).sum()}")

Features: 40
Feature list: ['owner_age', 'attitude_stable_business_environment', 'attitude_worried_shutdown', 'compliance_income_tax', 'perception_insurance_doesnt_cover_losses', 'perception_cannot_afford_insurance', 'personal_income', 'business_expenses', 'business_turnover', 'business_age_years', 'motor_vehicle_insurance', 'has_mobile_money', 'current_problem_cash_flow', 'has_cellphone', 'owner_sex', 'offers_credit_to_customers', 'attitude_satisfied_with_achievement', 'has_credit_card', 'keeps_financial_records', 'perception_insurance_companies_dont_insure_businesses_like_yours', 'perception_insurance_important', 'has_insurance', 'covid_essential_service', 'attitude_more_successful_next_year', 'problem_sourcing_money', 'marketing_word_of_mouth', 'has_loan_account', 'has_internet_banking', 'has_debit_card', 'future_risk_theft_stock', 'business_age_months', 'medical_insurance', 'funeral_insurance', 'motivation_make_more_money', 'uses_friends_family_savings', 'uses_informal_lender', 'co

In [6]:
def engineer_features(df):
    """
    Layer feature engineering on top of clean_and_preprocess output.
    Input df should already be preprocessed (all categoricals encoded).
    """
    df = df.copy()

    # --- Log-transform skewed financials ---
    for col in ["personal_income", "business_expenses", "business_turnover"]:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col].fillna(0))

    # --- Financial ratio features ---
    eps = 1e-6
    df["profit_proxy"]       = df["business_turnover"].fillna(0) - df["business_expenses"].fillna(0)
    df["log_profit_proxy"]   = np.log1p(np.maximum(df["profit_proxy"], 0))
    df["expense_ratio"]      = df["business_expenses"].fillna(0) / (df["business_turnover"].fillna(0) + eps)
    df["income_to_turnover"] = df["personal_income"].fillna(0) / (df["business_turnover"].fillna(0) + eps)
    df["income_to_expenses"] = df["personal_income"].fillna(0) / (df["business_expenses"].fillna(0) + eps)

    # --- Business age in total months ---
    df["total_business_months"] = (
        df["business_age_years"].fillna(0) * 12 +
        df["business_age_months"].fillna(0)
    )

    # --- Missing value flags for high-missingness columns ---
    high_missing = [
        "motor_vehicle_insurance", "has_mobile_money", "current_problem_cash_flow",
        "has_cellphone", "has_loan_account", "has_internet_banking", "has_debit_card",
        "future_risk_theft_stock", "medical_insurance", "funeral_insurance",
        "motivation_make_more_money", "uses_friends_family_savings",
        "uses_informal_lender", "business_age_months", "business_age_years"
    ]
    for col in high_missing:
        if col in df.columns:
            df[f"missing_{col}"] = df[col].isna().astype(int)

    # --- Count of financial products currently owned ---
    product_cols = [
        "motor_vehicle_insurance", "has_mobile_money", "has_credit_card",
        "has_loan_account", "has_internet_banking", "has_debit_card",
        "medical_insurance", "funeral_insurance"
    ]
    present = [c for c in product_cols if c in df.columns]
    df["n_financial_products"] = df[present].apply(
        lambda row: (row == 2).sum(), axis=1
    )

    # --- Positive attitude score ---
    attitude_cols = [
        "attitude_stable_business_environment",
        "attitude_more_successful_next_year",
        "attitude_satisfied_with_achievement"
    ]
    present_att = [c for c in attitude_cols if c in df.columns]
    df["attitude_score"] = df[present_att].apply(
        lambda row: (row == 1).sum(), axis=1
    )

    return df

# Apply both preprocessing and feature engineering
train_fe = clean_and_preprocess(train.drop(columns=[TARGET_COL, ID_COL]))
train_fe = engineer_features(train_fe)

test_fe  = clean_and_preprocess(test.drop(columns=[ID_COL]))
test_fe  = engineer_features(test_fe)

# Align columns
train_fe, test_fe = train_fe.align(test_fe, join="left", axis=1, fill_value=0)

# Encode target
y = le_target.transform(train[TARGET_COL])

# Impute
imputer = SimpleImputer(strategy="median")
X      = imputer.fit_transform(train_fe.values)
X_test = imputer.transform(test_fe.values)

feature_names = list(train_fe.columns)

print(f"Features after engineering: {len(feature_names)}")
print(f"NaN in X: {np.isnan(X).sum()}")
print(f"NaN in X_test: {np.isnan(X_test).sum()}")

Features after engineering: 66
NaN in X: 0
NaN in X_test: 0


## SMOTE setup

In [7]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(sampling_strategy="minority", random_state=SEED, k_neighbors=5)
print("SMOTE initialized — strategy: minority (oversample High class to match Medium)")

SMOTE initialized — strategy: minority (oversample High class to match Medium)


## Cross-validation setup

In [8]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def train_oof(model_cls, params, X, y, X_test, sm=None, label="Model"):
    """
    Generic OOF training loop.
    Applies SMOTE per fold if sm is provided.
    Returns oof_probs, test_probs.
    """
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]

        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        sw = compute_sample_weight("balanced", y_tr)
        m  = model_cls(**params, early_stopping_rounds=50)
        m.fit(X_tr, y_tr,
              sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])],
              verbose=False)

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score

print("CV utility ready.")

CV utility ready.


## XGBoost 

**XGBoost Optuna hyperparameter tuning**

In [9]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_objective(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "verbosity": 0,
        "use_label_encoder": False,
        "random_state": SEED,
        "tree_method": "hist",
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 20),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":             trial.suggest_float("gamma", 1e-4, 5.0, log=True),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        sw = compute_sample_weight("balanced", y_tr)
        m  = xgb.XGBClassifier(**params, early_stopping_rounds=50)
        m.fit(X_tr, y_tr,
              sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])],
              verbose=False)
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx])))
    return np.mean(scores)

print("Running Optuna hyperparameter search (50 trials)...")
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest Optuna F1 : {study.best_value:.4f}")
print(f"Best params    : {study.best_params}")

Running Optuna hyperparameter search (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best Optuna F1 : 0.8046
Best params    : {'n_estimators': 302, 'learning_rate': 0.0887785567490716, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.7844783677969637, 'colsample_bytree': 0.6714129799598364, 'gamma': 0.17236143498524512, 'reg_alpha': 0.3014557996862401, 'reg_lambda': 0.0013178580371381542}


 **XGB OOF training with best params**

In [10]:
best_params = study.best_params
best_params.update({
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "verbosity": 0,
    "use_label_encoder": False,
    "random_state": SEED,
    "tree_method": "hist",
})

print("Training XGBoost with best Optuna params + SMOTE...")
oof_xgb, test_xgb, xgb_score = train_oof(
    xgb.XGBClassifier, best_params, X, y, X_test,
    sm=sm, label="XGB"
)

Training XGBoost with best Optuna params + SMOTE...
  [XGB] Fold 1 F1: 0.8026
  [XGB] Fold 2 F1: 0.8328
  [XGB] Fold 3 F1: 0.7885
  [XGB] Fold 4 F1: 0.7971
  [XGB] Fold 5 F1: 0.8021

  [XGB] OOF Macro F1: 0.8050
              precision    recall  f1-score   support

        High       0.88      0.61      0.72       470
         Low       0.92      0.90      0.91      6280
      Medium       0.75      0.82      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.85      0.78      0.80      9618
weighted avg       0.87      0.86      0.86      9618



## LightGBM

**LightGBM Optuna tuning**

In [11]:
import lightgbm as lgb

def lgb_objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": SEED,
        "class_weight": "balanced",
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx])))
    return np.mean(scores)

print("Running Optuna for LightGBM (50 trials)...")
study_lgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest LGB Optuna F1 : {study_lgb.best_value:.4f}")
print(f"Best params        : {study_lgb.best_params}")

Running Optuna for LightGBM (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best LGB Optuna F1 : 0.8051
Best params        : {'n_estimators': 774, 'learning_rate': 0.032321327381520513, 'num_leaves': 140, 'max_depth': 6, 'min_child_samples': 65, 'subsample': 0.7014203310760537, 'colsample_bytree': 0.5030584902490122, 'reg_alpha': 0.0009821673055967987, 'reg_lambda': 0.0010309172101497146}


**LightGBM OOF training**

In [12]:
def train_oof_lgb(params, X, y, X_test, sm=None, label="LGB"):
    """OOF training loop for LightGBM (uses lgb callbacks instead of early_stopping_rounds)."""
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score

best_lgb_params = study_lgb.best_params
best_lgb_params.update({
    "objective": "multiclass", "num_class": 3,
    "metric": "multi_logloss", "verbosity": -1,
    "boosting_type": "gbdt", "random_state": SEED,
    "class_weight": "balanced"
})

print("Training LightGBM with best Optuna params + SMOTE...")
oof_lgb, test_lgb, lgb_score = train_oof_lgb(
    best_lgb_params, X, y, X_test, sm=sm, label="LGB"
)

Training LightGBM with best Optuna params + SMOTE...
  [LGB] Fold 1 F1: 0.8022
  [LGB] Fold 2 F1: 0.8278
  [LGB] Fold 3 F1: 0.7897
  [LGB] Fold 4 F1: 0.8048
  [LGB] Fold 5 F1: 0.8010

  [LGB] OOF Macro F1: 0.8054
              precision    recall  f1-score   support

        High       0.90      0.60      0.72       470
         Low       0.93      0.90      0.91      6280
      Medium       0.75      0.84      0.79      2868

    accuracy                           0.86      9618
   macro avg       0.86      0.78      0.81      9618
weighted avg       0.87      0.86      0.87      9618




## CatBoost 

**CatBoost Optuna tuning**

In [13]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "random_seed": SEED,
        "verbose": 0,
        "auto_class_weights": "Balanced",
        "iterations":          trial.suggest_int("iterations", 200, 1000),
        "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth":               trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=(X[val_idx], y[val_idx]),
              early_stopping_rounds=50)
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx]).flatten()))
    return np.mean(scores)

print("Running Optuna for CatBoost (50 trials)...")
study_cat = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_cat.optimize(cat_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest CAT Optuna F1 : {study_cat.best_value:.4f}")
print(f"Best params        : {study_cat.best_params}")

Running Optuna for CatBoost (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best CAT Optuna F1 : 0.7987
Best params        : {'iterations': 957, 'learning_rate': 0.039952845566170805, 'depth': 10, 'l2_leaf_reg': 0.0011799062523159302, 'bagging_temperature': 0.5978810668073862, 'random_strength': 0.5767509177705765}


**CatBoost OOF training**

In [14]:
def train_oof_cat(params, X, y, X_test, sm=None, label="CAT"):
    """OOF training loop for CatBoost."""
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=(X[val_idx], y[val_idx]),
              early_stopping_rounds=50)

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score

best_cat_params = study_cat.best_params
best_cat_params.update({
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "random_seed": SEED,
    "verbose": 0,
    "auto_class_weights": "Balanced"
})

print("Training CatBoost with best Optuna params + SMOTE...")
oof_cat, test_cat, cat_score = train_oof_cat(
    best_cat_params, X, y, X_test, sm=sm, label="CAT"
)

Training CatBoost with best Optuna params + SMOTE...
  [CAT] Fold 1 F1: 0.8009
  [CAT] Fold 2 F1: 0.8112
  [CAT] Fold 3 F1: 0.7801
  [CAT] Fold 4 F1: 0.7917
  [CAT] Fold 5 F1: 0.8096

  [CAT] OOF Macro F1: 0.7987
              precision    recall  f1-score   support

        High       0.78      0.66      0.72       470
         Low       0.93      0.88      0.90      6280
      Medium       0.72      0.83      0.77      2868

    accuracy                           0.85      9618
   macro avg       0.81      0.79      0.80      9618
weighted avg       0.86      0.85      0.86      9618



**Ensemble + threshold optimization + submission**

In [15]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def apply_thresholds(probs, thresholds):
    """Scale class probabilities by thresholds then argmax."""
    return np.argmax(probs / np.array(thresholds), axis=1)

def neg_macro_f1(thresholds, probs, y_true):
    return -macro_f1(y_true, apply_thresholds(probs, thresholds))

In [16]:
# -------------------------------------------------------------------------
# Weighted ensemble (weights proportional to OOF F1 scores)
# -------------------------------------------------------------------------
scores_arr = np.array([xgb_score, lgb_score, cat_score])
weights    = scores_arr / scores_arr.sum()

print(f"Ensemble weights — XGB: {weights[0]:.3f}, LGB: {weights[1]:.3f}, CAT: {weights[2]:.3f}")

oof_ensemble  = weights[0]*oof_xgb  + weights[1]*oof_lgb  + weights[2]*oof_cat
test_ensemble = weights[0]*test_xgb + weights[1]*test_lgb + weights[2]*test_cat

ensemble_score = macro_f1(y, np.argmax(oof_ensemble, axis=1))
print(f"\nEnsemble OOF F1 (default): {ensemble_score:.4f}")
print("\nPer-class breakdown (default):")
print(classification_report(y, np.argmax(oof_ensemble, axis=1), target_names=le_target.classes_))

# -------------------------------------------------------------------------
# Threshold optimization on ensemble
# -------------------------------------------------------------------------
print("Optimizing thresholds...")
result = minimize(
    neg_macro_f1,
    x0=[1.0, 1.0, 1.0],
    args=(oof_ensemble, y),
    method="Nelder-Mead",
    bounds=[(0.1, 2.0)] * 3,
    options={"maxiter": 5000, "xatol": 1e-5, "fatol": 1e-5}
)

best_thresholds = result.x
optimized_f1    = -result.fun

print(f"\nDefault   OOF F1 : {ensemble_score:.4f}")
print(f"Optimized OOF F1 : {optimized_f1:.4f}")
print(f"Thresholds — High: {best_thresholds[0]:.4f}, Low: {best_thresholds[1]:.4f}, Medium: {best_thresholds[2]:.4f}")

print("\nPer-class breakdown (optimized thresholds):")
print(classification_report(
    y, apply_thresholds(oof_ensemble, best_thresholds),
    target_names=le_target.classes_
))

# -------------------------------------------------------------------------
# Generate submission
# -------------------------------------------------------------------------
final_preds = le_target.inverse_transform(apply_thresholds(test_ensemble, best_thresholds))
submission  = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: final_preds})
submission.to_csv("submission_ensemble_clean.csv", index=False)

print(f"\nSaved: submission_ensemble_clean.csv")
print(f"Prediction distribution:\n{pd.Series(final_preds).value_counts()}")

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  XGBoost   OOF F1 : {xgb_score:.4f}")
print(f"  LightGBM  OOF F1 : {lgb_score:.4f}")
print(f"  CatBoost  OOF F1 : {cat_score:.4f}")
print(f"  Ensemble  OOF F1 : {ensemble_score:.4f}")
print(f"  Optimized OOF F1 : {optimized_f1:.4f}")
print("=" * 50)

Ensemble weights — XGB: 0.334, LGB: 0.334, CAT: 0.332

Ensemble OOF F1 (default): 0.7996

Per-class breakdown (default):
              precision    recall  f1-score   support

        High       0.85      0.61      0.71       470
         Low       0.92      0.89      0.91      6280
      Medium       0.74      0.83      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.84      0.78      0.80      9618
weighted avg       0.87      0.86      0.86      9618

Optimizing thresholds...

Default   OOF F1 : 0.7996
Optimized OOF F1 : 0.8082
Thresholds — High: 1.2534, Low: 0.6141, Medium: 1.1251

Per-class breakdown (optimized thresholds):
              precision    recall  f1-score   support

        High       0.88      0.60      0.71       470
         Low       0.88      0.98      0.93      6280
      Medium       0.88      0.70      0.78      2868

    accuracy                           0.88      9618
   macro avg       0.88      0.76      0.81     

In [17]:
submission

,ID,Target
0,ID_5EGLKX,Low
1,ID_4AI7RE,Low
2,ID_V9OB3M,Low
3,ID_6OI9DI,Low
4,ID_H2TN8B,Low
...,...,...
2400,ID_FX7XJZ,Low
2401,ID_XAL1LX,Low
2402,ID_UHBP0F,Medium
2403,ID_GKIKR2,Medium


In [18]:
submission.to_csv("submission_ensemble_clean.csv", index=False)
# shutil.move('/kaggle/working/submission_ensemble_clean.csv', '/kaggle/output/submission_ensemble_clean.csv')